In [38]:
import pandas as pd
import numpy as np
import spacy
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

In [42]:
TRAIN_PATH = 'INPUT/train.csv'
TEST_PATH  = 'INPUT/test.csv'
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

In [41]:
def add_spacy_vectors(df):
    vector_matrix = np.vstack(df['body'].apply(lambda x: nlp(x).vector))
    vector_columns = [f'vec_{i}' for i in range(vector_matrix.shape[1])]
    df[vector_columns] = pd.DataFrame(vector_matrix, index=df.index)
    return df
#df_train_with_vectors = add_spacy_vectors(df_train)
#df_encoded = pd.get_dummies(df_train_with_vectors, columns=['subreddit'])

encoder = OneHotEncoder(sparse_output=False)
encoded_array = encoder.fit_transform(df_train[['subreddit']])
encoded_df = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out(['subreddit']))
encoded_df.head()
# Concatenate with original DataFrame (optional: drop the original column)
#df_final = pd.concat([df.drop('category', axis=1), encoded_df], axis=1)


,subreddit_2007scape,subreddit_Android,subreddit_AskHistorians,subreddit_AskReddit,subreddit_AskTrumpSupporters,subreddit_AskWomen,subreddit_BlackPeopleTwitter,subreddit_CFB,subreddit_CanadaPolitics,subreddit_Christianity,...,subreddit_space,subreddit_spacex,subreddit_syriancivilwar,subreddit_technology,subreddit_television,subreddit_tifu,subreddit_videos,subreddit_whatisthisthing,subreddit_worldnews,subreddit_wow
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
df_train['subreddit'].value_counts()

subreddit
legaladvice            213
AskReddit              152
soccerstreams          139
personalfinance        125
relationships          106
                      ... 
OutOfTheLoop             1
LateStageCapitalism      1
fantasyfootball          1
changemyview             1
IAmA                     1
Name: count, Length: 100, dtype: int64

In [36]:
df_encoded.head()

,row_id,body,rule,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,vec_0,vec_1,...,subreddit_space,subreddit_spacex,subreddit_syriancivilwar,subreddit_technology,subreddit_television,subreddit_tifu,subreddit_videos,subreddit_whatisthisthing,subreddit_worldnews,subreddit_wow
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,-0.063569,0.214772,...,False,False,False,False,False,False,False,False,False,False
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,-0.107792,-0.047002,...,False,False,False,False,False,False,False,False,False,False
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,-0.041432,0.172439,...,False,False,False,False,False,False,False,False,False,False
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,0.026858,0.108277,...,False,False,False,False,False,False,False,False,False,False
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,-0.089873,0.105842,...,False,False,False,False,False,False,False,False,False,False


In [3]:
nlp = spacy.load('en_core_web_lg') 

In [22]:
def add_similarity(df, text_col, ref_col, new_col="similarity"):
    similarities = []

    for text, ref in zip(df[text_col], df[ref_col]):
        if all(isinstance(t, str) and t.strip() for t in [text, ref]):
            vec1 = nlp(text).vector.reshape(1, -1)
            vec2 = nlp(ref).vector.reshape(1, -1)
            sim = cosine_similarity(vec1, vec2)[0][0]
        else:
            sim = None
        similarities.append(sim)

    df[new_col] = similarities
    return df

In [23]:
df = add_similarity(df, "positive_example_1", "body", new_col="similarity")

In [26]:
df_train.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1


In [27]:
vector_matrix = np.vstack(df_train['body'].apply(lambda x: nlp(x).vector))

In [28]:
vector_columns = [f'vec_{i}' for i in range(vector_matrix.shape[1])]

In [29]:
df_train[vector_columns] = pd.DataFrame(vector_matrix, index=df_train.index)

/tmp/ipykernel_416/4095382594.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train[vector_columns] = pd.DataFrame(vector_matrix, index=df_train.index)
/tmp/ipykernel_416/4095382594.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train[vector_columns] = pd.DataFrame(vector_matrix, index=df_train.index)
/tmp/ipykernel_416/4095382594.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

In [31]:
df_train.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,vec_0,...,vec_290,vec_291,vec_292,vec_293,vec_294,vec_295,vec_296,vec_297,vec_298,vec_299
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,-0.063569,...,-0.315636,0.058424,-0.085667,-0.129196,0.216231,-0.196018,-0.150651,-0.109674,0.250563,0.179425
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,-0.107792,...,-0.073044,-0.081790,-0.049126,-0.017979,-0.070848,0.033082,-0.277724,-0.136934,-0.141995,0.024642
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,-0.041432,...,-0.184901,-0.013136,-0.168806,-0.051945,0.025863,-0.046664,-0.013163,-0.055355,0.123850,0.204677
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,0.026858,...,-0.053570,0.071715,0.011072,-0.084358,0.025244,-0.092253,0.067646,-0.030488,0.105881,-0.032873
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,-0.089873,...,-0.106687,-0.004151,-0.054862,-0.021714,-0.080993,-0.024750,-0.027726,-0.098017,-0.015035,0.118828


In [4]:
def get_vector(text):
    doc = nlp(text)
    return doc.vector

In [5]:
df = df_train
df['body_vector'] = df['body'].apply(get_vector)

In [9]:
def average_embedding(text):
    doc = nlp(text)
    # Filter out tokens without vectors (e.g., punctuation, stopwords sometimes)
    vectors = [token.vector for token in doc if token.has_vector and not token.is_space]
    
    if not vectors:
        return np.zeros(nlp.vocab.vectors_length)
    
    avg_vector = np.mean(vectors, axis=0)
    return avg_vector

df['body_vector'] = df['body'].apply(average_embedding)

In [10]:
df.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,body_vector
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,"[-0.06356867, 0.21477167, -0.24159677, -0.0171..."
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,"[-0.119768895, -0.052224364, -0.17059411, 0.01..."
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,"[-0.04143209, 0.17243934, -0.2742706, -0.16293..."
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,"[0.034182355, 0.13780683, -0.25608662, 0.15229..."
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,"[-0.11555123, 0.13608299, -0.17768349, -0.0085..."


In [11]:
df.shape

(2029, 10)

In [12]:
print(df['body_vector'].iloc[0].shape)

(300,)


In [13]:
df['positive_example_1_vector'] = df['positive_example_1'].apply(average_embedding)

In [15]:
def calculate_similarity(text1, text2):
    doc1 = nlp(text1)
    doc2 = nlp(text2)
    return doc1.similarity(doc2)

df['similarity'] = df.apply(lambda row: calculate_similarity(row['body'], row['positive_example_1']), axis=1)

/tmp/ipykernel_416/3395946515.py:4: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  return doc1.similarity(doc2)


In [16]:
df.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,body_vector,positive_example_1_vector,similarity
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,"[-0.06356867, 0.21477167, -0.24159677, -0.0171...","[-0.0688731, 0.24967559, -0.19641411, 0.047152...",0.903907
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,"[-0.119768895, -0.052224364, -0.17059411, 0.01...","[-0.1319535, 0.014230558, -0.17819946, -0.1099...",0.593202
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,"[-0.04143209, 0.17243934, -0.2742706, -0.16293...","[-0.08260104, 0.19201209, -0.2655245, -0.01527...",0.940281
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,"[0.034182355, 0.13780683, -0.25608662, 0.15229...","[-0.1558652, 0.3131602, -0.057391305, -0.08885...",0.653491
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,"[-0.11555123, 0.13608299, -0.17768349, -0.0085...","[-0.037172187, 0.1937645, -0.2013784, -0.04326...",0.752089


In [17]:
df['negative_example_1_vector'] = df['negative_example_1'].apply(average_embedding)

In [18]:
df['similarity_neg1'] = df.apply(lambda row: calculate_similarity(row['body'], row['negative_example_1']), axis=1)

/tmp/ipykernel_416/3395946515.py:4: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  return doc1.similarity(doc2)


In [19]:
df.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation,body_vector,positive_example_1_vector,similarity,negative_example_1_vector,similarity_neg1
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0,"[-0.06356867, 0.21477167, -0.24159677, -0.0171...","[-0.0688731, 0.24967559, -0.19641411, 0.047152...",0.903907,"[-0.13196085, 0.20436439, -0.06403447, -0.0019...",0.623155
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0,"[-0.119768895, -0.052224364, -0.17059411, 0.01...","[-0.1319535, 0.014230558, -0.17819946, -0.1099...",0.593202,"[0.004614919, 0.11464819, -0.07656955, 0.00145...",0.589201
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1,"[-0.04143209, 0.17243934, -0.2742706, -0.16293...","[-0.08260104, 0.19201209, -0.2655245, -0.01527...",0.940281,"[0.02523458, 0.16502906, -0.18383719, 0.001715...",0.895765
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1,"[0.034182355, 0.13780683, -0.25608662, 0.15229...","[-0.1558652, 0.3131602, -0.057391305, -0.08885...",0.653491,"[-0.27142, 0.047374, -0.17278, -0.029084, -0.2...",0.171015
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1,"[-0.11555123, 0.13608299, -0.17768349, -0.0085...","[-0.037172187, 0.1937645, -0.2013784, -0.04326...",0.752089,"[0.0863075, -0.17761075, -0.07848525, -0.10556...",0.604933
